In [3]:
import numpy as np 
import pandas as pd 

from sklearn.preprocessing import ( OneHotEncoder, StandardScaler)
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

# Models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Evaluation
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [4]:
df = pd.read_csv("../data/featured/telco_feature_engineered.csv")

In [5]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,TotalServices,HasSecurityBundle,AverageMonthlySpend,IsFamilyCustomer
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Yes,Electronic check,29.85,29.85,No,New Customers,2,1,29.850000,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,Mailed check,56.95,1889.50,No,2-4 Years,4,2,55.573529,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,New Customers,4,2,54.075000,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,Bank transfer (automatic),42.30,1840.75,No,2-4 Years,4,3,40.905556,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,New Customers,2,0,75.825000,0


In [6]:
x = df.drop('Churn', axis=1)
y = df['Churn']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state = 42,
    stratify=y
)

In [21]:
numerical_features = ['SeniorCitizen', 'tenure','MonthlyCharges', 'TotalCharges', 'TotalServices','HasSecurityBundle','AverageMonthlySpend']

categorical_features = ["gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "TenureGroup",
    "IsFamilyCustomer"]

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            'cat', 
            OneHotEncoder(handle_unknown="ignore"), 
            categorical_features
        )
    ]
)

In [23]:
result = {}

### Create an Evaluation Function

In [26]:
def evaluate_model(model, model_name):
    pipeline = Pipeline([
        ('Preprocessor', preprocessor),
        ('Classifier', model)
    ])
    
    #train
    pipeline.fit(X_train, y_train)

    #test
    y_pred = pipeline.predict(X_test)

    #calculate metrices
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label="Yes")
    recall = recall_score(y_test, y_pred, pos_label="Yes")
    f1 = f1_score(y_test, y_pred, pos_label="Yes")

    result[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'Recall': recall,
        'F1_Score': f1
    }

    print(f"\n{'='*50}")
    print(model_name)
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred))

### Train Models

In [27]:
evaluate_model(LogisticRegression(
    random_state=42,
    max_iter = 1000
),
"logistic Regression")


logistic Regression
              precision    recall  f1-score   support

          No       0.84      0.90      0.87      1035
         Yes       0.65      0.52      0.58       374

    accuracy                           0.80      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.79      0.80      0.79      1409



In [28]:
evaluate_model(DecisionTreeClassifier(random_state=42), 'Decision Tree')


Decision Tree
              precision    recall  f1-score   support

          No       0.82      0.81      0.82      1035
         Yes       0.50      0.52      0.51       374

    accuracy                           0.73      1409
   macro avg       0.66      0.67      0.66      1409
weighted avg       0.74      0.73      0.73      1409



In [29]:
evaluate_model(RandomForestClassifier(random_state=42), 'Random Forest')


Random Forest
              precision    recall  f1-score   support

          No       0.83      0.89      0.86      1035
         Yes       0.61      0.49      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [30]:
evaluate_model(
    KNeighborsClassifier(),
    "KNN"
)


KNN
              precision    recall  f1-score   support

          No       0.84      0.84      0.84      1035
         Yes       0.55      0.55      0.55       374

    accuracy                           0.76      1409
   macro avg       0.69      0.69      0.69      1409
weighted avg       0.76      0.76      0.76      1409



In [31]:
evaluate_model(
    SVC(random_state=42),
    "SVM"
)


SVM
              precision    recall  f1-score   support

          No       0.82      0.91      0.86      1035
         Yes       0.65      0.45      0.54       374

    accuracy                           0.79      1409
   macro avg       0.74      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409



In [32]:
result_df = pd.DataFrame(result).T

result_df

,accuracy,precision,Recall,F1_Score
logistic Regression,0.797729,0.649832,0.516043,0.575261
Decision Tree,0.732434,0.496203,0.524064,0.509753
Random Forest,0.779986,0.606667,0.486631,0.540059
KNN,0.760823,0.549598,0.548128,0.548862
SVM,0.790632,0.651341,0.454545,0.535433
